# Verify ABAC row filters on `*_with_abac` tables

**Goal:** Show that the same principal sees **all** rows on base tables (`patient_demographics`, `patient_claims`) but **fewer** rows on `patient_demographics_with_abac` / `patient_claims_with_abac` when `staff_patient_crosswalk` marks patients excluded for `CURRENT_USER()`.

Set **catalog** and **schema** to match your seed job (defaults match `databricks.yml`). Run all cells top to bottom.

In [ ]:
dbutils.widgets.text("catalog", "main", "Unity Catalog name")
dbutils.widgets.text("schema", "demo_dynamic_abac", "Schema name")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

def fq(name: str) -> str:
    return f"`{catalog}`.`{schema}`.`{name}`"

print(f"Using {catalog}.{schema}")

## Current user
Row filters compare `staff_patient_crosswalk.staff_email` to this value.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
who = spark.sql("SELECT current_user() AS current_user").collect()[0][0]
print("CURRENT_USER() =", who)

## Base tables (no row filter)
Expect full population: **10** patients, **100** claims (demo data).

In [ ]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {fq('patient_demographics')}").display()
spark.sql(f"SELECT COUNT(*) AS row_count FROM {fq('patient_claims')}").display()

## ABAC-protected copies (`*_with_abac`)
Same underlying data, but **Unity Catalog row filter** `check_patient_access` hides patients where this user appears in `staff_patient_crosswalk` with `is_excluded = 1` for that `patient_id`.

In [ ]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {fq('patient_demographics_with_abac')}").display()
spark.sql(f"SELECT COUNT(*) AS row_count FROM {fq('patient_claims_with_abac')}").display()

## Patients visible on base vs ABAC table
`EXCEPT` lists `patient_id` values present in the base table but **removed** from the ABAC view for **you** (empty if no exclusions apply to your user).

In [ ]:
spark.sql(f"""
SELECT patient_id
FROM {fq('patient_demographics')}
EXCEPT
SELECT patient_id
FROM {fq('patient_demographics_with_abac')}
ORDER BY patient_id
""").display()

## Claims rows only in the base table
Claims whose `patient_id` is hidden on the ABAC copy disappear here (count should match excluded claims for those patients).

In [ ]:
spark.sql(f"""
SELECT claim_id, patient_id
FROM {fq('patient_claims')}
EXCEPT
SELECT claim_id, patient_id
FROM {fq('patient_claims_with_abac')}
ORDER BY patient_id, claim_id
""").display()

## Crosswalk rows for your email (optional context)
Shows how `staff_email` lines up with `CURRENT_USER()` and which patients are flagged `is_excluded`.

In [ ]:
spark.sql(f"""
SELECT staff_id, patient_id, is_excluded, staff_email
FROM {fq('staff_patient_crosswalk')}
WHERE lower(trim(staff_email)) = lower(trim(current_user()))
ORDER BY staff_id, patient_id
""").display()